# Chapter 5.5: TokenizerManager & DetokenizerManager

## Learning Objectives

By the end of this notebook, you will:

1. **Understand why tokenization needs its own process** in SGLang
2. **Master TokenizerManager** architecture and concurrent tokenization
3. **Learn DetokenizerManager** incremental detokenization
4. **See how tokenization overlaps with GPU computation**
5. **Understand streaming detokenization challenges**
6. **Profile tokenization overhead** and optimization strategies

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.5_tokenizer.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.5_tokenizer.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why Separate Tokenizer Processes?

### The Problem

Tokenization is **CPU-bound** work that can bottleneck the GPU:

```
Without separate process:
┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐
│ Tokenize │  │   GPU    │  │Detokenize│  │ Tokenize │ ...
│ (CPU)    │  │ Forward  │  │ (CPU)    │  │ (CPU)    │
└──────────┘  └──────────┘  └──────────┘  └──────────┘
    |<- GPU idle ->|            |<- GPU idle ->|

With separate processes (overlap):
TokenizerMgr:  [Tok1][Tok2][Tok3][Tok4][Tok5]...
Scheduler/GPU: .....[GPU1][GPU2][GPU3][GPU4]...
DetokMgr:      ..........[Det1][Det2][Det3]...
                    ↑ Overlapped execution!
```

### Design Rationale

1. **CPU/GPU overlap**: While GPU runs forward pass, CPU tokenizes next requests
2. **Python GIL avoidance**: Separate process = separate GIL
3. **Scalability**: Can handle many concurrent tokenization tasks
4. **Isolation**: Tokenizer errors don't crash the scheduler

## 2. TokenizerManager Architecture

```python
# sglang/srt/managers/tokenizer_manager.py (simplified)

class TokenizerManager:
    """Manages tokenization for incoming requests.
    
    Runs in its own process. Responsibilities:
    1. Load and hold the tokenizer
    2. Apply chat templates to messages
    3. Tokenize input text
    4. Handle image preprocessing (multimodal)
    5. Forward tokenized requests to Scheduler
    6. Receive results from DetokenizerManager
    7. Match results to original requests and return to HTTP handler
    """
    
    def __init__(self, server_args, port_args):
        # Load tokenizer
        self.tokenizer = get_tokenizer(
            server_args.tokenizer_path or server_args.model_path,
            trust_remote_code=server_args.trust_remote_code
        )
        
        # ZMQ sockets for communication
        self.to_scheduler = zmq.Socket(PUSH)  # Send to scheduler
        self.from_detokenizer = zmq.Socket(PULL)  # Receive from detokenizer
        
        # Request tracking
        self.rid_to_state = {}  # rid -> RequestState
        
        # Chat template
        self.chat_template = get_chat_template(server_args)
    
    async def handle_request(self, request: GenerateReqInput):
        """Process an incoming generation request."""
        
        # Step 1: Apply chat template if messages format
        if request.messages:
            text = self.chat_template.apply(request.messages)
        else:
            text = request.text
        
        # Step 2: Tokenize
        input_ids = self.tokenizer.encode(text)
        
        # Step 3: Validate length
        if len(input_ids) > self.max_context_length:
            raise ValueError(f"Input too long: {len(input_ids)} > {self.max_context_length}")
        
        # Step 4: Handle multimodal (images/video)
        pixel_values = None
        if request.image_data:
            pixel_values = self.process_images(request.image_data)
        
        # Step 5: Create tokenized request
        tokenized = TokenizedGenerateReqInput(
            rid=request.rid,
            input_ids=input_ids,
            pixel_values=pixel_values,
            sampling_params=request.sampling_params,
        )
        
        # Step 6: Forward to scheduler
        self.to_scheduler.send_pyobj(tokenized)
```

In [ ]:
# Simulate TokenizerManager behavior

import time
import json
from typing import Dict, List, Optional
from dataclasses import dataclass, field

class SimulatedTokenizer:
    """Simulates a HuggingFace tokenizer."""
    
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        # Simple word-level tokenization for demo
        self._word_to_id = {}
        self._id_to_word = {}
        self._next_id = 1
    
    def encode(self, text: str) -> List[int]:
        """Tokenize text to token IDs."""
        words = text.lower().split()
        ids = []
        for word in words:
            if word not in self._word_to_id:
                self._word_to_id[word] = self._next_id
                self._id_to_word[self._next_id] = word
                self._next_id += 1
            ids.append(self._word_to_id[word])
        return ids
    
    def decode(self, ids: List[int], skip_special=True) -> str:
        """Decode token IDs to text."""
        words = [self._id_to_word.get(i, f"<unk:{i}>") for i in ids]
        return " ".join(words)


class ChatTemplate:
    """Simulates chat template application."""
    
    TEMPLATES = {
        "llama3": {
            "system": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{content}<|eot_id|>",
            "user": "<|start_header_id|>user<|end_header_id|>\n\n{content}<|eot_id|>",
            "assistant": "<|start_header_id|>assistant<|end_header_id|>\n\n{content}",
            "assistant_start": "<|start_header_id|>assistant<|end_header_id|>\n\n",
        },
        "chatml": {
            "system": "<|im_start|>system\n{content}<|im_end|>\n",
            "user": "<|im_start|>user\n{content}<|im_end|>\n",
            "assistant": "<|im_start|>assistant\n{content}<|im_end|>\n",
            "assistant_start": "<|im_start|>assistant\n",
        }
    }
    
    def __init__(self, template_name="llama3"):
        self.template = self.TEMPLATES[template_name]
    
    def apply(self, messages: List[dict]) -> str:
        """Apply chat template to messages."""
        result = ""
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            if role in self.template:
                result += self.template[role].format(content=content)
        # Add assistant start marker
        result += self.template["assistant_start"]
        return result


# Demonstrate chat template application
template = ChatTemplate("llama3")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
]

formatted = template.apply(messages)
print("Chat Template Application")
print("=" * 60)
print(f"Input messages: {json.dumps(messages, indent=2)}")
print(f"\nFormatted text:\n{formatted}")

# Tokenize
tokenizer = SimulatedTokenizer()
tokens = tokenizer.encode(formatted)
print(f"\nToken IDs ({len(tokens)} tokens): {tokens[:20]}...")
print(f"Decoded back: {tokenizer.decode(tokens[:10])}...")

## 3. DetokenizerManager — Incremental Detokenization

### The Challenge

Detokenization is not simply the reverse of tokenization. Several challenges:

1. **Token boundaries**: A single character might span multiple tokens
2. **Unicode**: Multi-byte characters can be split across tokens
3. **BPE merges**: Token boundaries depend on context
4. **Streaming**: Need to output text incrementally as tokens arrive

```
Problem: incremental detokenization

Tokens: ["Hello", " world", "!", " I", "'m", " hap", "py"]
                                              ↑
              After receiving "hap", we can't output it yet
              because the next token might merge with it!
              ("hap" + "py" = "happy", not "hap py")
```

### Solution: Incremental Detokenization

```python
# sglang/srt/managers/detokenizer_manager.py (simplified)

class DetokenizerManager:
    """Handles incremental detokenization.
    
    Key insight: decode the full sequence so far and take the diff
    from the previous decode. This handles all edge cases.
    """
    
    def __init__(self, server_args, port_args):
        self.tokenizer = get_tokenizer(server_args.model_path)
        
        # Track state per request
        self.request_states = {}  # rid -> DetokenizeState
    
    def process_batch_output(self, batch_output: BatchTokenIDOut):
        """Process a batch of token ID outputs."""
        results = []
        
        for rid, output_ids, finished in zip(
            batch_output.rids,
            batch_output.output_ids,
            batch_output.finished
        ):
            state = self.request_states.get(rid)
            if state is None:
                state = DetokenizeState(rid=rid)
                self.request_states[rid] = state
            
            # Add new token(s)
            state.all_output_ids.extend(output_ids)
            
            # Incremental decode
            new_text = self._incremental_decode(state)
            
            results.append(BatchStrOut(
                rid=rid,
                output_str=new_text,
                finished=finished,
            ))
            
            if finished:
                del self.request_states[rid]
        
        return results
    
    def _incremental_decode(self, state):
        """Decode only the new portion of text."""
        # Decode all tokens so far
        full_text = self.tokenizer.decode(
            state.all_output_ids,
            skip_special_tokens=True
        )
        
        # Take the diff from last decode
        new_text = full_text[len(state.prev_text):]
        state.prev_text = full_text
        
        return new_text
```

In [ ]:
# Demonstrate incremental detokenization

@dataclass
class DetokenizeState:
    rid: str
    all_output_ids: List[int] = field(default_factory=list)
    prev_text: str = ""
    prev_len: int = 0

class IncrementalDetokenizer:
    """Demonstrates incremental detokenization."""
    
    def __init__(self):
        # Simulated vocabulary
        self.vocab = {
            1: "The", 2: " capital", 3: " of", 4: " France",
            5: " is", 6: " Paris", 7: ".", 8: " It",
            9: " is", 10: " known", 11: " for", 12: " the",
            13: " Ei", 14: "ffel", 15: " Tower", 16: ".",
        }
    
    def decode_all(self, ids: List[int]) -> str:
        return "".join(self.vocab.get(i, f"<{i}>") for i in ids)
    
    def incremental_decode(self, state: DetokenizeState, new_ids: List[int]) -> str:
        """Decode incrementally, handling token boundary issues."""
        state.all_output_ids.extend(new_ids)
        
        # Full decode
        full_text = self.decode_all(state.all_output_ids)
        
        # Diff from previous
        new_text = full_text[len(state.prev_text):]
        state.prev_text = full_text
        
        return new_text

# Simulate streaming detokenization
detok = IncrementalDetokenizer()
state = DetokenizeState(rid="req-001")

# Tokens arrive one at a time
token_stream = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

print("Incremental Detokenization Demo")
print("=" * 60)
print(f"{'Step':>5s} {'Token ID':>10s} {'New Text':>20s} {'Full Text So Far'}")
print("-" * 80)

for i, token_id in enumerate(token_stream):
    new_text = detok.incremental_decode(state, [token_id])
    full_text = state.prev_text
    
    # Highlight the tricky case with "Ei" + "ffel"
    marker = " <-- subword!" if token_id in [13, 14] else ""
    
    print(f"{i+1:5d} {token_id:10d} {repr(new_text):>20s} {repr(full_text)}{marker}")

In [ ]:
# Demonstrate the overlap between tokenization, GPU, and detokenization

class PipelineTimingSimulator:
    """Simulate timing of the tokenize-compute-detokenize pipeline."""
    
    def __init__(self, tokenize_ms=2.0, compute_ms=20.0, detokenize_ms=0.5):
        self.tokenize_ms = tokenize_ms
        self.compute_ms = compute_ms
        self.detokenize_ms = detokenize_ms
    
    def simulate_sequential(self, num_requests: int) -> float:
        """No overlap: tokenize, compute, detokenize sequentially."""
        total = 0
        for _ in range(num_requests):
            total += self.tokenize_ms + self.compute_ms + self.detokenize_ms
        return total
    
    def simulate_pipelined(self, num_requests: int) -> float:
        """Overlapped: processes run in parallel."""
        # First request: full pipeline
        # Subsequent: overlap tokenize(N+1) with compute(N)
        if num_requests == 0:
            return 0
        
        # Time = first_tokenize + N * max(compute, tokenize) + last_detokenize
        # Since compute >> tokenize, bottleneck is compute
        total = self.tokenize_ms  # First tokenize
        total += num_requests * max(self.compute_ms, self.tokenize_ms)  # Pipelined
        total += self.detokenize_ms  # Last detokenize
        return total

sim = PipelineTimingSimulator()

print("Pipeline Overlap Analysis")
print("=" * 60)
print(f"Per-request timing: tokenize={sim.tokenize_ms}ms, "
      f"compute={sim.compute_ms}ms, detokenize={sim.detokenize_ms}ms")
print()

for n in [1, 5, 10, 50, 100]:
    seq_time = sim.simulate_sequential(n)
    pipe_time = sim.simulate_pipelined(n)
    speedup = seq_time / pipe_time
    print(f"  {n:3d} requests: Sequential={seq_time:.0f}ms, "
          f"Pipelined={pipe_time:.0f}ms, Speedup={speedup:.2f}x")

## 4. Streaming Detokenization

For streaming responses, the DetokenizerManager must:
1. Receive tokens one at a time from the Scheduler
2. Incrementally decode to text
3. Send text chunks to the HTTP handler for SSE

### The UTF-8 Challenge

```python
# Some tokens produce incomplete UTF-8 byte sequences
# Example with CJK characters:
# Token 50000 might encode bytes [0xE4, 0xB8]  (first 2 bytes of a 3-byte char)
# Token 50001 might encode bytes [0xAD]        (last byte)
# Together: 0xE4 0xB8 0xAD = Chinese char U+4E2D

# Solution: buffer incomplete UTF-8 sequences
class StreamingDetokenizer:
    def decode_token(self, token_id):
        # Decode to bytes
        new_bytes = self.tokenizer.convert_ids_to_bytes(token_id)
        self.byte_buffer.extend(new_bytes)
        
        # Try to decode as UTF-8
        try:
            text = self.byte_buffer.decode('utf-8')
            self.byte_buffer = bytearray()  # Clear buffer
            return text
        except UnicodeDecodeError:
            # Incomplete UTF-8 sequence, wait for more bytes
            return ""
```

In [ ]:
# Simulate the complete tokenization pipeline

class TokenizationPipeline:
    """Full tokenization pipeline simulation."""
    
    def __init__(self):
        self.tokenizer = SimulatedTokenizer()
        self.chat_template = ChatTemplate("llama3")
    
    def process_chat_request(self, messages, max_tokens=50):
        """Process a chat completion request end-to-end."""
        print("Tokenization Pipeline")
        print("=" * 60)
        
        # Step 1: Apply chat template
        t0 = time.time()
        formatted = self.chat_template.apply(messages)
        t1 = time.time()
        print(f"1. Chat template applied ({(t1-t0)*1000:.2f}ms)")
        print(f"   Formatted length: {len(formatted)} chars")
        
        # Step 2: Tokenize
        t2 = time.time()
        input_ids = self.tokenizer.encode(formatted)
        t3 = time.time()
        print(f"\n2. Tokenized ({(t3-t2)*1000:.2f}ms)")
        print(f"   Tokens: {len(input_ids)}")
        print(f"   First 10 IDs: {input_ids[:10]}")
        
        # Step 3: Validate
        max_context = 4096
        remaining = max_context - len(input_ids)
        effective_max_tokens = min(max_tokens, remaining)
        print(f"\n3. Validated")
        print(f"   Context budget: {len(input_ids)}/{max_context} used")
        print(f"   Effective max_tokens: {effective_max_tokens}")
        
        # Step 4: Simulate generation + detokenization
        print(f"\n4. Generation + Incremental Detokenization:")
        detok_state = DetokenizeState(rid="demo")
        detok = IncrementalDetokenizer()
        
        # Simulate generating 5 tokens
        generated_tokens = [1, 2, 3, 4, 5]
        full_output = ""
        for i, tok_id in enumerate(generated_tokens):
            new_text = detok.incremental_decode(detok_state, [tok_id])
            full_output += new_text
            print(f"   Token {i+1}: id={tok_id} -> '{new_text}'")
        
        print(f"\n5. Final output: '{full_output}'")
        return full_output

# Run the pipeline
pipeline = TokenizationPipeline()
pipeline.process_chat_request([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
])

## 5. Performance Analysis

### Tokenization Overhead

Tokenization typically takes 0.1-5ms per request, depending on:
- Input length
- Tokenizer type (BPE, SentencePiece, etc.)
- Chat template complexity
- Number of images (for multimodal)

In [ ]:
# Profile tokenization overhead

import time

def benchmark_tokenization(tokenizer, texts, num_iterations=100):
    """Benchmark tokenization speed."""
    results = []
    
    for text in texts:
        times = []
        for _ in range(num_iterations):
            start = time.perf_counter()
            tokens = tokenizer.encode(text)
            end = time.perf_counter()
            times.append((end - start) * 1000)  # ms
        
        avg_ms = sum(times) / len(times)
        results.append({
            "text_len": len(text),
            "num_tokens": len(tokens),
            "avg_ms": avg_ms,
            "tokens_per_ms": len(tokens) / avg_ms if avg_ms > 0 else 0,
        })
    
    return results

# Generate test texts of varying lengths
tokenizer = SimulatedTokenizer()
base_text = "The quick brown fox jumps over the lazy dog. "

test_texts = [
    base_text * 1,       # ~10 words
    base_text * 10,      # ~100 words
    base_text * 100,     # ~1000 words
    base_text * 500,     # ~5000 words
]

results = benchmark_tokenization(tokenizer, test_texts)

print("Tokenization Performance Benchmark")
print("=" * 70)
print(f"{'Text Length':>12s} {'Tokens':>8s} {'Avg Time':>10s} {'Tokens/ms':>12s}")
print("-" * 70)
for r in results:
    print(f"{r['text_len']:12,d} {r['num_tokens']:8d} "
          f"{r['avg_ms']:9.3f}ms {r['tokens_per_ms']:11.0f}")

print("\nNote: Real tokenizers (BPE/SentencePiece) are slower due to")
print("merge operations. Typical speeds: 10K-100K tokens/sec.")

## 6. Source Code Walkthrough: Key Methods

```python
# sglang/srt/managers/tokenizer_manager.py — event loop

class TokenizerManager:
    async def loop_for_forward(self):
        """Main event loop for the TokenizerManager."""
        while True:
            # Receive from HTTP handlers (via asyncio queue)
            recv_obj = await self.recv_from_http.get()
            
            if isinstance(recv_obj, GenerateReqInput):
                # Generation request
                await self._handle_generate(recv_obj)
            elif isinstance(recv_obj, EmbeddingReqInput):
                # Embedding request
                await self._handle_embedding(recv_obj)
            elif isinstance(recv_obj, FlushCacheReq):
                # Cache flush
                self.to_scheduler.send_pyobj(recv_obj)
    
    async def loop_for_result(self):
        """Receive results from DetokenizerManager."""
        while True:
            result = await self.from_detokenizer.recv_pyobj()
            
            for rid, output_str, finished in zip(
                result.rids, result.output_strs, result.finished
            ):
                state = self.rid_to_state.get(rid)
                if state:
                    if state.is_streaming:
                        # Send chunk to streaming queue
                        await state.stream_queue.put(
                            StreamChunk(text=output_str, finished=finished)
                        )
                    elif finished:
                        # Send complete result
                        state.result = output_str
                        state.event.set()  # Wake up waiting handler
                    
                    if finished:
                        del self.rid_to_state[rid]
```

In [ ]:
# Complete TokenizerManager + DetokenizerManager simulation

import asyncio
import queue

class FullPipelineSimulation:
    """Simulate the full tokenize-schedule-detokenize pipeline."""
    
    def __init__(self):
        self.tokenizer = SimulatedTokenizer()
        self.chat_template = ChatTemplate("llama3")
        self.request_log = []
    
    def process_request(self, messages, max_tokens=10, stream=False):
        """Process a complete request."""
        rid = f"req-{len(self.request_log)+1:03d}"
        log = {"rid": rid, "steps": []}
        
        # TokenizerManager
        t0 = time.perf_counter()
        formatted = self.chat_template.apply(messages)
        input_ids = self.tokenizer.encode(formatted)
        t1 = time.perf_counter()
        log["steps"].append(("TokenizerManager", "tokenize", f"{(t1-t0)*1000:.3f}ms", f"{len(input_ids)} tokens"))
        
        # Scheduler (simulated)
        log["steps"].append(("Scheduler", "prefix_match", "0.01ms", "cache lookup"))
        log["steps"].append(("Scheduler", "allocate", "0.01ms", f"{len(input_ids)} KV slots"))
        
        # ModelRunner (simulated)
        generated_ids = list(range(100, 100 + max_tokens))
        log["steps"].append(("ModelRunner", "extend", "15.0ms", f"prefill {len(input_ids)} tokens"))
        for i in range(max_tokens):
            log["steps"].append(("ModelRunner", f"decode_{i+1}", "1.5ms", f"token {generated_ids[i]}"))
        
        # DetokenizerManager
        detok_state = DetokenizeState(rid=rid)
        detok = IncrementalDetokenizer()
        
        if stream:
            for tok_id in generated_ids:
                new_text = detok.incremental_decode(detok_state, [tok_id])
                log["steps"].append(("DetokenizerManager", "stream_chunk", "0.05ms", repr(new_text)))
        else:
            full_text = detok.incremental_decode(detok_state, generated_ids)
            log["steps"].append(("DetokenizerManager", "detokenize", "0.1ms", repr(full_text)))
        
        self.request_log.append(log)
        return log
    
    def display_trace(self, log):
        print(f"\nRequest {log['rid']} Trace")
        print("=" * 70)
        for component, action, timing, detail in log["steps"]:
            detail_str = detail if len(detail) < 40 else detail[:37] + "..."
            print(f"  {component:<22s} {action:<15s} [{timing:>8s}] {detail_str}")

sim = FullPipelineSimulation()
log = sim.process_request(
    [{"role": "user", "content": "Hello!"}],
    max_tokens=5,
    stream=True
)
sim.display_trace(log)

## 7. Summary

### Key Takeaways

1. **TokenizerManager runs in a separate process** to overlap CPU tokenization with GPU computation
2. **Chat templates** are applied before tokenization to format messages correctly
3. **DetokenizerManager handles incremental detokenization** by decoding the full sequence and taking diffs
4. **Streaming detokenization** must handle incomplete UTF-8 sequences and subword tokens
5. **Tokenization overhead** is typically 0.1-5ms per request, small compared to GPU compute
6. **The pipeline design** enables CPU/GPU overlap for higher throughput

### Next Chapter

Chapter 5.6 will explore **ModelRunner & TpModelWorker** — how the actual model forward pass is executed.

## Exercises

### Exercise 1: Custom Chat Template
Implement a custom chat template for a model that uses a different format.

### Exercise 2: Streaming Detokenizer
Implement a streaming detokenizer that handles UTF-8 boundary issues with a byte buffer.

### Exercise 3: Profile Real Tokenizer
Install the `transformers` library and profile tokenization speed for different models:
- Llama tokenizer (BPE)
- T5 tokenizer (SentencePiece)
Compare encoding and decoding speeds.

### Exercise 4: Batch Tokenization
Design and implement a batched tokenization strategy that processes multiple requests simultaneously for better CPU utilization.

In [ ]:
# Exercise 2 starter code

class StreamingUTF8Detokenizer:
    """TODO: Handle UTF-8 boundary issues in streaming detokenization."""
    
    def __init__(self):
        self.byte_buffer = bytearray()
        self.text_so_far = ""
    
    def add_bytes(self, new_bytes: bytes) -> str:
        """Add new bytes and return any complete UTF-8 text."""
        self.byte_buffer.extend(new_bytes)
        
        # TODO: Try to decode as much UTF-8 as possible
        # Keep incomplete sequences in the buffer
        # Return only the newly decoded text
        
        pass

print("Exercise 2: Implement StreamingUTF8Detokenizer.add_bytes()")